# Actividad 2

In [3]:
# Paso 1: Cargar las librerías necesarias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configuración
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True, precision=4)
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')


# URL del dataset de costos de seguro médico (Fuente: Kaggle/UC Irvine)
# Utilizaremos una URL raw para cargarlo directamente
_URL_ = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

# Cargar el dataset
df = pd.read_csv(_URL_)

# --- Diagnóstico Inicial ---
print("--- Diagnóstico Inicial del Dataset ---")
print(df.info())
print("\nPrimeras 5 filas:")
print(df.head())

# --- Paso 2: Preprocesamiento (Codificación de Variables Categóricas) ---
# Se utiliza One-Hot Encoding para las variables nominales: sex, smoker, region.

# Codificación de variables categóricas
df_encoded = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)

# Renombrar columnas para facilitar el modelado (ej: 'sex_male' -> 'is_male')
df_encoded.columns = df_encoded.columns.str.replace('[ -]', '_', regex=True)
df_encoded.rename(columns={'sex_male': 'is_male',
                           'smoker_yes': 'is_smoker'}, inplace=True)


# Definición de Variables
X = df_encoded.drop('charges', axis=1) # Variables predictoras
y = df_encoded['charges']              # Variable objetivo (costo)

# División en Entrenamiento y Prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Estandarización de Variables Numéricas (age, bmi, children)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

cols_to_scale = ['age', 'bmi', 'children']
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print("\n--- Vista Previa del DataFrame Codificado y Escalado ---")
print(X_train_scaled.head())


# --- PASO 3: MODELADO Y COMPARACIÓN DE REGRESIÓN ---

resultados_modelos = {}

# 3.1 MODELO BASE: REGRESIÓN LINEAL MÚLTIPLE
model_base = LinearRegression()
model_base.fit(X_train_scaled, y_train)
y_pred_base = model_base.predict(X_test_scaled)

r2_base_test = r2_score(y_test, y_pred_base)
mae_base_test = mean_absolute_error(y_test, y_pred_base)
r2_base_train = r2_score(y_train, model_base.predict(X_train_scaled))

resultados_modelos['Lineal Base'] = {'R2_Train': r2_base_train, 'R2_Test': r2_base_test, 'MAE': mae_base_test}

print("\n--- 3.1: REGRESIÓN LINEAL BASE ---")
print(f"R2 Entrenamiento: {r2_base_train:.4f} | R2 Prueba: {r2_base_test:.4f}")
print(f"MAE Prueba: ${mae_base_test:,.2f}")


# --- 3.2 REGRESIÓN POLINOMIAL (GRADO 2) ---

# Crear características polinomiales
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

# Entrenar modelo Polinomial
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)
y_pred_poly = model_poly.predict(X_test_poly)

r2_poly_test = r2_score(y_test, y_pred_poly)
mae_poly_test = mean_absolute_error(y_test, y_pred_poly)
r2_poly_train = r2_score(y_train, model_poly.predict(X_train_poly))

resultados_modelos['Polinomial'] = {'R2_Train': r2_poly_train, 'R2_Test': r2_poly_test, 'MAE': mae_poly_test}

print("\n--- 3.2: REGRESIÓN POLINOMIAL (GRADO 2) ---")
print(f"R2 Entrenamiento: {r2_poly_train:.4f} | R2 Prueba: {r2_poly_test:.4f}")
print(f"MAE Prueba: ${mae_poly_test:,.2f}")

# Detección de Sobreajuste: Si R2_Train >> R2_Test, hay sobreajuste.

# --- 3.3 REGULARIZACIÓN (RIDGE L2 y LASSO L1) ---

# Aplicar Regularización Ridge (L2)
# Alpha alto para penalizar fuertemente
model_ridge = Ridge(alpha=1000)
model_ridge.fit(X_train_poly, y_train)
y_pred_ridge = model_ridge.predict(X_test_poly)

r2_ridge_test = r2_score(y_test, y_pred_ridge)
mae_ridge_test = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge_train = r2_score(y_train, model_ridge.predict(X_train_poly))

resultados_modelos['Ridge L2'] = {'R2_Train': r2_ridge_train, 'R2_Test': r2_ridge_test, 'MAE': mae_ridge_test}

print("\n--- 3.3.a: REGULARIZACIÓN RIDGE (L2) ---")
print(f"R2 Entrenamiento: {r2_ridge_train:.4f} | R2 Prueba: {r2_ridge_test:.4f}")
print(f"MAE Prueba: ${mae_ridge_test:,.2f}")


# Aplicar Regularización Lasso (L1)
# Alpha bajo para no eliminar todos los coeficientes
model_lasso = Lasso(alpha=200)
model_lasso.fit(X_train_poly, y_train)
y_pred_lasso = model_lasso.predict(X_test_poly)

r2_lasso_test = r2_score(y_test, y_pred_lasso)
mae_lasso_test = mean_absolute_error(y_test, y_pred_lasso)
r2_lasso_train = r2_score(y_train, model_lasso.predict(X_train_poly))

resultados_modelos['Lasso L1'] = {'R2_Train': r2_lasso_train, 'R2_Test': r2_lasso_test, 'MAE': mae_lasso_test}

print("\n--- 3.3.b: REGULARIZACIÓN LASSO (L1) ---")
print(f"R2 Entrenamiento: {r2_lasso_train:.4f} | R2 Prueba: {r2_lasso_test:.4f}")
print(f"MAE Prueba: ${mae_lasso_test:,.2f}")


# --- PASO 4: ANÁLISIS COMPARATIVO FINAL ---
print("\n--- 4. ANÁLISIS COMPARATIVO DE MODELOS ---")
df_comparacion = pd.DataFrame(resultados_modelos).T
print(df_comparacion.sort_values(by='R2_Test', ascending=False))

# Hallazgo clave: El modelo con el mayor R2_Test y menor MAE será el mejor.


--- Diagnóstico Inicial del Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB
None

Primeras 5 filas:
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520

--- Vista Previa del DataFrame 